In [1]:
pip install pyreadstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 24.8 MB/s eta 0:00:00


# Data processing


In [3]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4"
import pandas as pd
import pyreadstat
import numpy as np
from sklearn.model_selection import KFold, cross_val_score, StratifiedKFold, train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, SVMSMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import matplotlib.pyplot as plt

from scipy.stats import spearmanr


# Load the SAS file
def load_data(file_path):
    df, meta = pyreadstat.read_sas7bdat(file_path)
    return df

# Delete Proxy Skipped data
def drop_proxy_skipped(df):
    df = df.drop(df[df.AJ31 == -2].index)
    return df

def drop_dont_know(df, target_columns):
    df = df.copy()
    for col in target_columns:
        df = df[df[col] != 3]
        df[col] = df[col].replace({1: 0, 2: 1})
    return df

def multi_target_encode_kfold(data, features, target, n_splits=5, seed=42):
    df = data.copy()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    encoding_maps = {}

    for feature in features:
        col_encoded = feature + '_te'
        df[col_encoded] = np.nan

        for train_idx, valid_idx in kf.split(df):
            train_fold, valid_fold = df.iloc[train_idx], df.iloc[valid_idx]
            means = train_fold.groupby(feature)[target].mean()
            df.loc[df.index[valid_idx], col_encoded] = df.loc[df.index[valid_idx], feature].map(means)

        encoding_maps[feature] = df.groupby(feature)[target].mean().to_dict()

    return df, encoding_maps


def preprocess_features(df, target_column):
    df = df.copy()


    social2_map = {
        -1: 0,
        0: 0,
        1: 1,
        2: 2,
    }
    df['SOCIAL2'] = df['SOCIAL2'].map(social2_map)


    numcig_map = {
        1: 0,
        2: 1,
        3: 2,
        4: 3,
        5: 4,
        6: 5
    }
    df['NUMCIG'] = df['NUMCIG'].map(numcig_map)


    ac208_map = {
        -1: 3,  # Never
        1: 0,
        2: 1,
        3: 2
    }
    df['AC208'] = df['AC208'].map(ac208_map)


    ae15a_map = {
        1: 2,
        2: 1,
        3: 0,
        -1: 0
    }
    df['AE15A'] = df['AE15A'].map(ae15a_map)


    df['MARIT'] = df['MARIT'].replace({1: 0, 2: 1, 3: 2})


    aj29_map = {
        1: 5,
        2: 4,
        3: 3,
        4: 2,
        5: 1
    }
    df['AJ29'] = df['AJ29'].map(aj29_map)


    ac117v2_days_map = {
        -1: 0,
        1: 0,
        2: 1.5,
        3: 4,
        4: 7.5,
        5: 15,
        6: 25,
        7: 30
    }
    df['AC117V2'] = df['AC117V2'].map(ac117v2_days_map)


    srage_map = {
        18: 21.5,
        26: 27.5,
        30: 32,
        35: 37,
        40: 42,
        45: 47,
        50: 52,
        55: 57,
        60: 62,
        65: 67,
        70: 72,
        75: 77,
        80: 82,
        85: 87
    }
    df['SRAGE_P1'] = df['SRAGE_P1'].map(srage_map)


    rbmi_map = {
        1: 0,  # UNDERWEIGHT
        2: 1,  # NORMAL
        3: 2,  # OVERWEIGHT
        4: 3  # OBESE
    }
    df['RBMI'] = df['RBMI'].map(rbmi_map)


    high_card_features = ['OCCMAIN2', 'OMBSRR_P1', 'HOUSETYPE']
    for col in high_card_features:
        df[col] = df[col].replace(99, 15)

    df_encoded, encoding_maps = multi_target_encode_kfold(
        data=df,
        features=high_card_features,
        target=target_column
    )

    for col in high_card_features:
        df[col] = df_encoded[col]

    return df, encoding_maps


def compute_spearman_top10_and_save(df, target_cols, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for target in target_cols:
        print(f"\n🔍 Processing for {target}")

        df_corr = df.dropna(subset=[target])
        feature_cols = [col for col in df_corr.columns if col not in DISEASE_COLUMNS]

        corr_results = {
            feat: spearmanr(df_corr[feat], df_corr[target])[0]
            for feat in feature_cols
        }

        top10_features = pd.Series(corr_results).abs().sort_values(ascending=False).head(10).index.tolist()
        df_top10 = df_corr[top10_features + [target]].copy()


        if target == 'AB29V2':
            df_top10[target] = df_top10[target].apply(lambda x: 1 if x in [1, 3] else 0)
        elif target == 'AB154':
            df_top10 = df_top10[df_top10[target] != 3]
            df_top10[target] = df_top10[target].apply(lambda x: 1 if x == 1 else 0)
        else:
            df_top10[target] = df_top10[target].apply(lambda x: 1 if x == 1 else 0)

        save_path = os.path.join(output_dir, f"{target}_df.csv")
        df_top10.to_csv(save_path, index=False)
        print(f"✅ Saved to {save_path}")

if __name__ == "__main__":
    PATH = 'adult.sas7bdat'

    # List of feature columns
    SELECTED_FEATURES = [
        "AC117V2", "AC174", "AC207", "AC208", "AC212", "AC81C",
        "AE15A", "AF81",
        "AJ29", "AJ30", "AJ31", "AJ32", "AJ33", "AJ34",
        "DSTRS12", "DSTRSYR", "BINGE30",
        "ESMKCUR", "HOUSETYPE", "AQ1", "AQ12",
        "MARCUR", "MAREXPOSE", "MARIT",
        "NUMCIG", "OCCMAIN2", "OMBSRR_P1",
        "RBMI", "SMKEXPOSE",
        "SMOKING", "SOCIAL2", "SRAGE_P1", "SRSEX",
        "TOBCUR", "UR_CLRT4", "WGHTK_P"
    ]

    DISEASE_COLUMNS = ["AB154", "AB17", "AB22V2", "AB29V2", "AB34"]

    df = load_data(PATH)

    for disease in DISEASE_COLUMNS:
      data = df[SELECTED_FEATURES + [disease]].copy()
      data, _ = preprocess_features(data, disease)
      data = drop_proxy_skipped(data)
      compute_spearman_top10_and_save(data, [disease], 'save_file')



🔍 Processing for AB154
✅ Saved to save_file/AB154_df.csv

🔍 Processing for AB17
✅ Saved to save_file/AB17_df.csv

🔍 Processing for AB22V2
✅ Saved to save_file/AB22V2_df.csv

🔍 Processing for AB29V2
✅ Saved to save_file/AB29V2_df.csv

🔍 Processing for AB34
✅ Saved to save_file/AB34_df.csv


In [4]:
AB34_df = pd.read_csv('save_file/AB34_df.csv')
AB22V2_df =  pd.read_csv('save_file/AB22V2_df.csv')
AB17_df =  pd.read_csv('save_file/AB17_df.csv')
AB29V2_df = pd.read_csv('save_file/AB29V2_df.csv')
AB154_df = pd.read_csv('save_file/AB154_df.csv')

In [5]:
AB17_df.head(10)

,AQ1,DSTRSYR,SOCIAL2,AF81,RBMI,AJ33,AJ31,DSTRS12,AJ29,WGHTK_P,AB17
0,2.0,6.0,0.0,2.0,3,4.0,4.0,2.0,2.0,108.86,0
1,2.0,2.0,0.0,2.0,1,4.0,4.0,2.0,1.0,70.31,0
2,2.0,9.0,1.0,1.0,2,3.0,3.0,2.0,3.0,68.04,1
3,1.0,12.0,2.0,1.0,3,4.0,3.0,2.0,1.0,108.86,0
4,2.0,3.0,0.0,2.0,1,4.0,4.0,2.0,1.0,73.94,0
5,2.0,0.0,0.0,2.0,1,5.0,5.0,2.0,1.0,48.99,0
6,2.0,5.0,0.0,2.0,3,4.0,4.0,2.0,3.0,117.03,0
7,2.0,0.0,0.0,2.0,3,5.0,5.0,2.0,1.0,81.65,0
8,2.0,2.0,0.0,2.0,1,5.0,3.0,2.0,1.0,68.04,0
9,2.0,8.0,0.0,2.0,1,5.0,1.0,2.0,5.0,58.97,0


In [ ]:
print(AB34_df.columns)
print(AB22V2_df.columns)
print(AB17_df.columns)
print(AB29V2_df.columns)
print(AB154_df.columns)

Index(['SRAGE_P1', 'OCCMAIN2', 'SMOKING', 'SRSEX', 'AC174', 'BINGE30',
       'MAREXPOSE', 'WGHTK_P', 'AC212', 'AC81C', 'AB34'],
      dtype='object')
Index(['SRAGE_P1', 'RBMI', 'WGHTK_P', 'AC208', 'OCCMAIN2', 'BINGE30', 'AC212',
       'MAREXPOSE', 'AC207', 'SRSEX', 'AB22V2'],
      dtype='object')
Index(['AQ1', 'DSTRSYR', 'SOCIAL2', 'AF81', 'RBMI', 'AJ33', 'AJ31', 'DSTRS12',
       'AJ29', 'WGHTK_P', 'AB17'],
      dtype='object')
Index(['SRAGE_P1', 'RBMI', 'OCCMAIN2', 'WGHTK_P', 'SMOKING', 'MAREXPOSE',
       'AC212', 'AC208', 'AC174', 'DSTRSYR', 'AB29V2'],
      dtype='object')
Index(['SRAGE_P1', 'RBMI', 'OCCMAIN2', 'MARIT', 'SMOKING', 'WGHTK_P',
       'MAREXPOSE', 'AC81C', 'HOUSETYPE', 'AC212', 'AB154'],
      dtype='object')


# Check Data Leakage

In [6]:

def detect_leakage(df, target_col, threshold=0.9):
    print(f"🔍 check（with {target_col} correlation ≥ {threshold}）:")


    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if target_col in num_cols:
        num_cols.remove(target_col)


    corr = df[num_cols + [target_col]].corr(method='spearman')[target_col]
    corr = corr.drop(target_col)

    leakage_risks = corr[abs(corr) >= threshold]

    if not leakage_risks.empty:
        print("⚠️ possible data leakage")
        print(leakage_risks.sort_values(key=abs, ascending=False))
    else:
        print("✅ there is no data leakage")


detect_leakage(AB22V2_df, 'AB22V2')
detect_leakage(AB17_df, 'AB17')
detect_leakage(AB29V2_df, 'AB29V2')
detect_leakage(AB34_df, 'AB34')
detect_leakage(AB154_df, 'AB154')

🔍 check（with AB22V2 correlation ≥ 0.9）:
✅ there is no data leakage
🔍 check（with AB17 correlation ≥ 0.9）:
✅ there is no data leakage
🔍 check（with AB29V2 correlation ≥ 0.9）:
✅ there is no data leakage
🔍 check（with AB34 correlation ≥ 0.9）:
✅ there is no data leakage
🔍 check（with AB154 correlation ≥ 0.9）:
✅ there is no data leakage


# Decision Tree

In [8]:
import json
import pandas as pd
import numpy as np

In [7]:
import os, joblib, json
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score, make_scorer
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline


Path("save_file").mkdir(parents=True, exist_ok=True)

FILE_LIST = [
    "save_file/AB29V2_df.csv",
    "save_file/AB34_df.csv",
    "save_file/AB22V2_df.csv",
    "save_file/AB17_df.csv",
    "save_file/AB154_df.csv"
]

PARAM_GRID = {
    "clf__max_depth":        [None, 3, 5, 8],
    "clf__min_samples_leaf": [1, 2, 4],
    "clf__ccp_alpha":        [0.0, 0.0005, 0.001]
}
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


dt_results = []
RANDOM_STATE = 42

for csv_path in FILE_LIST:
    df = pd.read_csv(csv_path)
    target = df.columns[-1]
    feature_cols = df.columns[:-1].tolist()
    X, y = df[feature_cols], df[target]

    pipe = Pipeline([
        ("smote", SMOTE(random_state=42)),
        ("clf", DecisionTreeClassifier(class_weight="balanced", random_state=42))
    ])

    grid = GridSearchCV(
        pipe, PARAM_GRID, cv=CV, n_jobs=-1, scoring=make_scorer(f1_score)
    )
    grid.fit(X, y)
    best_pipe = grid.best_estimator_
    print(f"[{target}] best params:", grid.best_params_, "  F1 (CV)=", grid.best_score_)


    model_file   = f"save_file/new_{target.lower()}_model_new.pkl"
    feature_file = f"save_file/new_{target.lower()}_features.pkl"
    joblib.dump(best_pipe, model_file)
    joblib.dump(feature_cols, feature_file)
    print(f"✅ Saved {model_file} & {feature_file}")


    y_pred  = best_pipe.predict(X)
    y_proba = best_pipe.predict_proba(X)[:, 1]
    acc   = accuracy_score(y, y_pred)
    f1    = f1_score(y, y_pred)
    rec   = recall_score(y, y_pred)
    prec  = precision_score(y, y_pred)
    auc   = roc_auc_score(y, y_proba)

    dt_results.append({
        "target": target,
        "accuracy": round(acc, 4),
        "auc": round(auc, 4),
        "f1_score": round(f1, 4),
        "precision": round(prec, 4),
        "recall": round(rec, 4)
    })

final_df = pd.DataFrame(dt_results)
print("\n📊 Decision‑Tree Evaluations:")
print(final_df.sort_values(by=["target", "accuracy"], ascending=[True, False]))


[AB29V2] best params: {'clf__ccp_alpha': 0.0005, 'clf__max_depth': 5, 'clf__min_samples_leaf': 1}   F1 (CV)= 0.6440751414365518
✅ Saved save_file/new_ab29v2_model_new.pkl & save_file/new_ab29v2_features.pkl
[AB34] best params: {'clf__ccp_alpha': 0.001, 'clf__max_depth': 5, 'clf__min_samples_leaf': 1}   F1 (CV)= 0.3018621343521924
✅ Saved save_file/new_ab34_model_new.pkl & save_file/new_ab34_features.pkl
[AB22V2] best params: {'clf__ccp_alpha': 0.0, 'clf__max_depth': 3, 'clf__min_samples_leaf': 1}   F1 (CV)= 0.29140353278814823
✅ Saved save_file/new_ab22v2_model_new.pkl & save_file/new_ab22v2_features.pkl
[AB17] best params: {'clf__ccp_alpha': 0.0, 'clf__max_depth': 3, 'clf__min_samples_leaf': 1}   F1 (CV)= 0.22786539757488802
✅ Saved save_file/new_ab17_model_new.pkl & save_file/new_ab17_features.pkl
[AB154] best params: {'clf__ccp_alpha': 0.0, 'clf__max_depth': 3, 'clf__min_samples_leaf': 1}   F1 (CV)= 0.3479007620420462
✅ Saved save_file/new_ab154_model_new.pkl & save_file/new_ab154_f

# Naive Bayes

In [9]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

file_list = [
    "save_file/AB154_df.csv"
]

nb_results = []
RANDOM_STATE = 30

for file in file_list:
    df = pd.read_csv(file)
    target_col = df.columns[-1]
    X = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )


    nb_pipe = Pipeline([
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('clf', GaussianNB())
    ])


    nb_pipe.fit(X_train, y_train)


    y_pred  = nb_pipe.predict(X_test)
    y_proba = nb_pipe.predict_proba(X_test)[:, 1]

    nb_results.append({
        "target":     target_col,
        "accuracy":   round(accuracy_score(y_test, y_pred), 4),
        "auc":        round(roc_auc_score(y_test, y_proba), 4),
        "f1_score":   round(f1_score(y_test, y_pred), 4),
        "precision":  round(precision_score(y_test, y_pred), 4),
        "recall":     round(recall_score(y_test, y_pred), 4)
    })


final_df = pd.DataFrame(nb_results)
print("\n📊 Naive Bayes Evaluations:")
print(final_df.sort_values(by=["target", "accuracy"], ascending=[True, False]))



📊 Naive Bayes Evaluations:
  target  accuracy     auc  f1_score  precision  recall
0  AB154    0.5097  0.6083    0.3937     0.2747  0.6946


In [ ]:
joblib.dump(nb_pipe, 'save_file/nb_model.pkl')

['save_file/nb_model.pkl']

# CNN

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.metrics import AUC, Recall
import pickle

# List of files to process
file_list = [
    "save_file/AB29V2_df.csv",
    "save_file/AB34_df.csv",
    "save_file/AB22V2_df.csv",
    "save_file/AB17_df.csv",
    "save_file/AB154_df.csv"
]

mlp_results = []
RANDOM_STATE = 42

for file in file_list:

    df = pd.read_csv(file)
    target_col = df.columns[-1]
    X = df.drop(columns=[target_col])
    y = df[target_col]

    if "AB22V2_df.csv" in file:
      features = X.columns.tolist()
      with open("ab22v2_features.pkl", "wb") as f:
        pickle.dump(features, f)
      print("saved AB22V2 features to ab22v2_features.pkl")



    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )

    X_train_np = X_train.values
    X_test_np  = X_test.values
    y_train_np = y_train.values
    y_test_np  = y_test.values


    model = Sequential()
    model.add(Dense(64, activation='relu', input_shape=(X_train_np.shape[1],)))
    model.add(Dropout(0.2))
    model.add(Dense(48, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(32, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(16, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy', Recall(name="recall"), AUC(name='auc')])


    class_weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train_np)
    class_weight_dict = {0: class_weights[0], 1: class_weights[1]}


    history = model.fit(
        X_train_np, y_train_np,
        validation_split=0.2,
        epochs=40,
        batch_size=32,
        class_weight=class_weight_dict,
        verbose=0
    )

    y_proba = model.predict(X_test_np).flatten()
    y_pred = (y_proba >= 0.5).astype(int)


    mlp_results.append({
        "target":     target_col,
        "accuracy":   round(accuracy_score(y_test_np, y_pred), 4),
        "auc":        round(roc_auc_score(y_test_np, y_proba), 4),
        "f1_score":   round(f1_score(y_test_np, y_pred), 4),
        "precision":  round(precision_score(y_test_np, y_pred), 4),
        "recall":     round(recall_score(y_test_np, y_pred), 4)
    })


final_df = pd.DataFrame(mlp_results)
print("\n📊 MLP Evaluations:")
print(final_df.sort_values(by=["target", "accuracy"], ascending=[True, False]))


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


136/136 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


136/136 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
saved AB22V2 features to ab22v2_features.pkl


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


136/136 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


136/136 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

📊 MLP Evaluations:
   target  accuracy     auc  f1_score  precision  recall
4   AB154    0.4805  0.6627    0.4299     0.2872  0.8546
3    AB17    0.6841  0.6106    0.3228     0.2623  0.4196
2  AB22V2    0.6624  0.7452    0.3502     0.2333  0.7023
0  AB29V2    0.6774  0.7485    0.6430     0.5813  0.7193
1    AB34    0.6024  0.7818    0.2801     0.1683  0.8354


In [ ]:
model.save('disease_files/model_file/mlp_model.h5')

In [12]:
import pandas as pd

data_json = {
    "target": ["AB154", "AB22V2", "AB29V2", "AB34"],
    "accuracy": [0.5097, 0.6624, 0.6831, 0.8182],
    "auc":      [0.6083, 0.7452, 0.7469, 0.7555],
    "f1_score": [0.3937, 0.3502, 0.6439, 0.3253],
    "precision": [0.2747, 0.2333, 0.5893, 0.2477],
    "recall":    [0.6946, 0.7023, 0.7097, 0.4738]
}

df = pd.DataFrame(data)
print(df)


   target  accuracy     auc  f1_score  precision  recall
0   AB154    0.5097  0.6083    0.3937     0.2747  0.6946
1  AB22V2    0.6624  0.7452    0.3502     0.2333  0.7023
2  AB29V2    0.6831  0.7469    0.6439     0.5893  0.7097
3    AB34    0.8182  0.7555    0.3253     0.2477  0.4738


In [ ]:

json_input = '''
{"SRSEX":"1",
"SRAGE_P1":"58",
"OMBSRR_P1":"2",
"MARIT":"0",
"OCCMAIN2":"2",
"UR_CLRT4":"1",
"RBMI":"2",
"WGHTK_P":"100",
"AC117V2":"7",
"AC118V2":"3",
"AC174":"0","SMKCUR30":"2","AC207":"2","AC208":"3","BINGE30":"2",
"AC212":"2","AC81C":"2","ESMKCUR":"2","SMKEXPOSE":"2",
"SMOKING":"1","AE15A":"0",
"NUMCIG":"1","AF81":"2","AJ29":"1",
"AJ30":"1","AJ31":"1","AJ32":"1",
"AJ33":"1","AJ34":"1","DSTRS12":"2",
"HOUSETYPE":"2","AQ1":"1","AQ12":"2",
"MARCUR":"2","MAREXPOSE":"2","SOCIAL2":"0",
"TOBCUR":"2"}
'''


input_dict = json.loads(json_input)
input_df = pd.DataFrame([input_dict]).astype(float)



disease_map = {
    "AB34": ("new_ab34_model_new.pkl", "ab34_features.pkl"),
    "AB22V2": ("AB22V2_model.h5", "ab22v2_features.pkl"),
    "AB29V2": ("new_ab29v2_model_new.pkl", "ab29v2_features.pkl"),
    "AB154" : ("nb_model.pkl", "ab154_features.pkl")

}

disease_name_map = {
    "AB34": "HEART DISEASE",
    "AB22V2": "DIABETES",
    "AB29V2": "HIGH BLOOD PRESSURE",
    "AB154": "HIGH CHOLESTEROL"
}



all_required_features = set()
for _, feature_file in disease_map.values():
    features = joblib.load(f"save_file/{feature_file}")
    all_required_features.update(features)

for feat in all_required_features:
    if feat not in input_df.columns:
        input_df[feat] = 0


results = {}

for disease, (model_file, feature_file) in disease_map.items():
    model_path = f"save_file/{model_file}"
    feature_path = f"save_file/{feature_file}"

    feature_list = joblib.load(feature_path)
    X = input_df[feature_list]

    ext = os.path.splitext(model_file)[1]

    if ext == '.pkl':
        model = joblib.load(model_path)
        print(type(model))
        prob = model.predict_proba(X)[0][1]
    elif ext == '.h5':
        keras_model = load_model(model_path)
        print(model_path)
        prob = float(keras_model.predict(X, verbose=0)[0][0])

    results[disease] = round(prob, 4)

print("✅ Predicted Probabilities:")
for disease_code, prob in results.items():
    name = disease_name_map.get(disease_code, disease_code)
    print(f"{name}: {prob:.4f}")

<class 'imblearn.pipeline.Pipeline'>
save_file/AB22V2_model.h5
<class 'imblearn.pipeline.Pipeline'>
<class 'imblearn.pipeline.Pipeline'>
✅ Predicted Probabilities:
HEART DISEASE: 0.6535
DIABETES: 0.6133
HIGH BLOOD PRESSURE: 0.6103
HIGH CHOLESTEROL: 0.6835
